# Start Partition Comparison

This notebook compares the available start partition algorithms for the Dense Graph Partition experiments.

The comparison is performed separately for:

- Powerlaw and Erdős–Rényi graphs,
- sparse and dense instances,
- small and large instances.

Only two aggregated metrics are reported:

- **mean relative to best**: mean quotient between the solution density of an algorithm and the best density found on the same instance;
- **mean runtime**: mean runtime in seconds.

A value close to `1.0` for the relative solution quality indicates that an algorithm produces solutions close to the best available result.

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

In [11]:
RESULTS_FILE = Path("../results/experiment1/raw_results.csv")

ALGORITHM_ORDER = [
    "singleton",
    "matching",
    "maximum_matching",
    "maximum_matching_edge_cover",
    "high_degree_first_matching",
    "high_degree_product_matching",
    "kapoce",
    "leiden",
]

GRAPH_ORDER = ["powerlaw", "er"]
REGIME_ORDER = ["sparse", "dense"]
SIZE_ORDER = ["small", "large"]

## Load experiment results

The raw experiment results are loaded and checked for the columns required by this analysis.

In [12]:
raw = pd.read_csv(RESULTS_FILE)

required_columns = {
    "graph_type",
    "regime",
    "size_class",
    "algorithm",
    "relative_to_best",
    "runtime",
}

missing_columns = required_columns.difference(raw.columns)

if missing_columns:
    raise ValueError(
        "The result file is missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

print(f"Loaded {len(raw):,} result rows.")
print("Algorithms:", ", ".join(sorted(raw["algorithm"].unique())))
raw.head()

Loaded 16,000 result rows.
Algorithms: high_degree_first_matching, high_degree_product_matching, kapoce, leiden_mdgp, matching, maximum_matching, maximum_matching_edge_cover, singleton


,dataset,size_class,graph_type,regime,instance,n,m,edge_density,algorithm,density,num_clusters,max_cluster_size,avg_cluster_size,runtime,relative_to_best,is_best
0,powerlaw_sparse_small,small,powerlaw,sparse,small_powerlaw_sparse_014_n60,60,221,0.1249,singleton,0.0,60,1,1.0000,0.0001,inf,False
1,powerlaw_sparse_small,small,powerlaw,sparse,small_powerlaw_sparse_014_n60,60,221,0.1249,matching,13.5,33,2,1.8182,0.0003,1.3111,False
2,powerlaw_sparse_small,small,powerlaw,sparse,small_powerlaw_sparse_014_n60,60,221,0.1249,maximum_matching,15.0,30,2,2.0000,0.0021,1.1800,False
3,powerlaw_sparse_small,small,powerlaw,sparse,small_powerlaw_sparse_014_n60,60,221,0.1249,maximum_matching_edge_cover,15.0,30,2,2.0000,0.0017,1.1800,False
4,powerlaw_sparse_small,small,powerlaw,sparse,small_powerlaw_sparse_014_n60,60,221,0.1249,high_degree_first_matching,12.5,35,2,1.7143,0.0004,1.4160,False


## Aggregate solution quality and runtime

For every combination of graph type, density regime, size class, and start partition algorithm, the notebook calculates:

1. the mean relative solution quality;
2. the mean runtime in seconds.

Each instance contributes one observation to the corresponding group.

In [13]:
summary = (
    raw
    .groupby(
        ["graph_type", "regime", "size_class", "algorithm"],
        as_index=False,
        observed=True,
    )
    .agg(
        mean_relative_to_best=("relative_to_best", "mean"),
        mean_runtime_seconds=("runtime", "mean"),
    )
)

summary["graph_type"] = pd.Categorical(
    summary["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)
summary["regime"] = pd.Categorical(
    summary["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)
summary["size_class"] = pd.Categorical(
    summary["size_class"],
    categories=SIZE_ORDER,
    ordered=True,
)

present_algorithms = summary["algorithm"].unique().tolist()
algorithm_order = [
    algorithm
    for algorithm in ALGORITHM_ORDER
    if algorithm in present_algorithms
]
algorithm_order += sorted(
    set(present_algorithms).difference(algorithm_order)
)

summary["algorithm"] = pd.Categorical(
    summary["algorithm"],
    categories=algorithm_order,
    ordered=True,
)

summary = (
    summary
    .sort_values(["graph_type", "size_class", "regime", "algorithm"])
    .reset_index(drop=True)
)

summary

,graph_type,regime,size_class,algorithm,mean_relative_to_best,mean_runtime_seconds
0,powerlaw,sparse,small,singleton,inf,0.000030
1,powerlaw,sparse,small,matching,1.410530,0.000336
2,powerlaw,sparse,small,maximum_matching,1.114775,0.019037
3,powerlaw,sparse,small,maximum_matching_edge_cover,1.111345,0.017708
4,powerlaw,sparse,small,high_degree_first_matching,1.489544,0.001751
...,...,...,...,...,...,...
59,er,dense,large,maximum_matching_edge_cover,1.330450,1.148428
60,er,dense,large,high_degree_first_matching,1.375946,0.064794
61,er,dense,large,high_degree_product_matching,1.375946,0.033409
62,er,dense,large,kapoce,1.000000,0.229682


In [14]:
final_table = summary[
    [
        "graph_type",
        "size_class",
        "regime",
        "algorithm",
        "mean_relative_to_best",
        "mean_runtime_seconds",
    ]
].copy()

final_table["mean_relative_to_best"] = (
    final_table["mean_relative_to_best"].round(4)
)
final_table["mean_runtime_seconds"] = (
    final_table["mean_runtime_seconds"].round(5)
)

final_table

,graph_type,size_class,regime,algorithm,mean_relative_to_best,mean_runtime_seconds
0,powerlaw,small,sparse,singleton,inf,0.00003
1,powerlaw,small,sparse,matching,1.4105,0.00034
2,powerlaw,small,sparse,maximum_matching,1.1148,0.01904
3,powerlaw,small,sparse,maximum_matching_edge_cover,1.1113,0.01771
4,powerlaw,small,sparse,high_degree_first_matching,1.4895,0.00175
...,...,...,...,...,...,...
59,er,large,dense,maximum_matching_edge_cover,1.3304,1.14843
60,er,large,dense,high_degree_first_matching,1.3759,0.06479
61,er,large,dense,high_degree_product_matching,1.3759,0.03341
62,er,large,dense,kapoce,1.0000,0.22968


## LaTeX helper functions

The following functions format algorithm names and numerical values for the thesis table. Values are truncated rather than rounded, matching the formatting used in the move-operator notebook.

In [15]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_algorithm(algorithm: str) -> str:
    return r"\texttt{" + algorithm.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{truncate_number(value, decimals):.{decimals}f}"

## Build LaTeX comparison table

The table is grouped by graph type and dataset configuration. Within each dataset group:

- the highest mean relative solution quality is printed in bold

Ties are highlighted for all affected algorithms.

In [16]:
def make_start_partition_latex_table(
        df: pd.DataFrame,
        graph_type: str,
        caption: str,
        label: str,
) -> str:
    dataset_order = [
        ("small", "sparse"),
        ("small", "dense"),
        ("large", "sparse"),
        ("large", "dense"),
    ]

    graph_df = df[df["graph_type"] == graph_type].copy()

    if graph_df.empty:
        raise ValueError(
            f"No results available for graph type '{graph_type}'."
        )

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        (
            r"\begin{tabular}{"
            r"p{2cm}"
            r"p{6.3cm}"
            r"p{3.1cm}"
            r"p{2.1cm}"
            r"}"
        ),
        r"\toprule",
        (
            r"Datensatz "
            r"& Startpartition "
            r"& Mittlere relative Lösungsqualität "
            r"& Mittlere Laufzeit (s) \\"
        ),
        r"\midrule",
    ]

    nonempty_datasets = [
        (size_class, regime)
        for size_class, regime in dataset_order
        if not graph_df[
            (graph_df["size_class"] == size_class)
            & (graph_df["regime"] == regime)
            ].empty
    ]

    for dataset_index, (size_class, regime) in enumerate(
            nonempty_datasets
    ):
        part = graph_df[
            (graph_df["size_class"] == size_class)
            & (graph_df["regime"] == regime)
            ].copy()

        part["algorithm"] = pd.Categorical(
            part["algorithm"],
            categories=algorithm_order,
            ordered=True,
        )
        part = part.sort_values("algorithm")

        best_quality = part["mean_relative_to_best"].min()

        dataset_label = f"{size_class} {regime}"

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset_label}}}"
                if row_index == 0
                else ""
            )

            quality = format_number(
                row.mean_relative_to_best,
                4,
            )
            runtime = format_number(
                row.mean_runtime_seconds,
                5,
            )

            if np.isclose(
                    row.mean_relative_to_best,
                    best_quality,
            ):
                quality = rf"\textbf{{{quality}}}"

            lines.append(
                f"{dataset_cell} "
                f"& {latex_algorithm(str(row.algorithm))} "
                f"& {quality} "
                f"& {runtime} \\\\"
            )

        if dataset_index < len(nonempty_datasets) - 1:
            lines.append(r"\cmidrule(l){1-4}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [17]:
powerlaw_latex = make_start_partition_latex_table(
    final_table,
    graph_type="powerlaw",
    caption=(
        "Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung."
    ),
    label="tab:start_partition_powerlaw",
)

print(powerlaw_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung.}
\label{tab:start_partition_powerlaw}
\begin{tabular}{p{2cm}p{6.3cm}p{3.1cm}p{2.1cm}}
\toprule
Datensatz & Startpartition & Mittlere relative Lösungsqualität & Mittlere Laufzeit (s) \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & inf & 0.00003 \\
 & \texttt{matching} & 1.4105 & 0.00034 \\
 & \texttt{maximum\_matching} & 1.1148 & 0.01904 \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.1113 & 0.01771 \\
 & \texttt{high\_degree\_first\_matching} & 1.4895 & 0.00175 \\
 & \texttt{high\_degree\_product\_matching} & 1.4895 & 0.00084 \\
 & \texttt{kapoce} & \textbf{1.0008} & 0.00989 \\
 & \texttt{leiden\_mdgp} & 1.0445 & 0.00120 \\
\cmi

In [18]:
er_latex = make_start_partition_latex_table(
    final_table,
    graph_type="er",caption=(
        "Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Erdős-Rényi-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung."
    ),
    label="tab:start_partition_er",
)

print(er_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Erdős-Rényi-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Verfahrens. Ein Wert von 1 entspricht der besten gefundenen Lösung.}
\label{tab:start_partition_er}
\begin{tabular}{p{2cm}p{6.3cm}p{3.1cm}p{2.1cm}}
\toprule
Datensatz & Startpartition & Mittlere relative Lösungsqualität & Mittlere Laufzeit (s) \\
\midrule
\multirow{8}{*}{small sparse} & \texttt{singleton} & inf & 0.00003 \\
 & \texttt{matching} & 1.2709 & 0.00037 \\
 & \texttt{maximum\_matching} & 1.1661 & 0.01961 \\
 & \texttt{maximum\_matching\_edge\_cover} & 1.1627 & 0.01880 \\
 & \texttt{high\_degree\_first\_matching} & 1.3646 & 0.00316 \\
 & \texttt{high\_degree\_product\_matching} & 1.3646 & 0.00092 \\
 & \texttt{kapoce} & \textbf{1.0000} & 0.01235 \\
 & \texttt{leiden\_mdgp} & 1.1168 & 0.00120 \\
\cmidru